# Part 8 · Notebook 03 — Survivorship bias and the tear sheet

**Sessions:** S5 (Backtesting biases & pitfalls) · S6 (Performance analysis) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Build a point-in-time universe and measure what survivorship bias adds.
2. Compute the maximum drawdown and the longest time under water.
3. Add Sortino and Calmar to a tear sheet, and read five strategies side by side.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. Survivorship bias

Sixty stocks over ten years; some go bust and are **delisted** (their price series ends), a few list later. If you backtest on the stocks that exist *today*, you have silently removed every loser from the past. A point-in-time universe on date `d` holds the symbols with `start <= d` and (`end` is missing or `d < end`).

In [ ]:
closes, membership = p.stock_universe()
membership[membership.end.notna() | (membership.start > closes.index[0])]

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def members(membership, date):
    d = pd.Timestamp(date)
    m = ...                                       # ✍️ the rows alive on d (listed, and not yet delisted)
    return sorted(m["symbol"])

dates = ["2014-01-02", "2017-06-30", "2020-03-02", "2023-12-29"]
res = [p.attempt(members, membership, d) for d in dates]
mine = [len(x) for x in res] if all(x is not Ellipsis for x in res) else Ellipsis
mine = p.check("point-in-time universe sizes", mine, [len(p.members(membership, d)) for d in dates])
dict(zip(dates, mine))

In [ ]:
rets = closes.pct_change(fill_method=None)
survivors = [s for s in closes.columns if s in p.members(membership, closes.index[-1])]
biased = rets[survivors].mean(axis=1).fillna(0)                               # today's members, all history
pit = rets.apply(lambda row: row[[s for s in p.members(membership, row.name) if pd.notna(row.get(s))]].mean(), axis=1).fillna(0)
for name, r in [("survivors only (biased)", biased), ("point-in-time universe", pit)]:
    print(f"{name:26s} CAGR {p.tear_sheet(r)['cagr']:+.2%}  Sharpe {p.tear_sheet(r)['sharpe']:.2f}")

## 2. Drawdown and time under water

Compound the returns into equity, compare each point with the running peak (starting from 1.0), and report the **worst** drawdown (a negative number) and the **longest run** of consecutive bars below a previous peak.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def max_drawdown(returns):
    eq = np.cumprod(1 + np.asarray(returns, dtype=float))
    peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    dd = eq / peak - 1
    longest = run = 0
    for x in dd:
        ...                                       # ✍️ count consecutive bars under water, keep the longest run
    return float(dd.min()), int(longest)

strats = p.strategy_returns()
mine = [p.attempt(max_drawdown, strats[s]) for s in strats]
mine = p.check("max_drawdown", mine, [p.max_drawdown(strats[s]) for s in strats])
pd.DataFrame(mine, index=strats.columns, columns=["max drawdown", "longest under water (days)"])

Five strategies with **known** annual Sharpe ratios (0.8, 0.6, 0.5, 0.4, 0.3). Even the best one spends over a year under water. Tell your future self before going live.

## 3. Sortino and Calmar

* **Sortino** = mean / downside deviation × √252, where downside deviation = `√mean(min(r, 0)²)` (only losses count as risk);
* **Calmar** = CAGR / |max drawdown|.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def sortino_calmar(returns):
    r = np.asarray(returns, dtype=float)
    cagr = np.prod(1 + r) ** (252 / r.size) - 1
    mdd = p.max_drawdown(r)[0]
    sortino = ...                                 # ✍️
    calmar = ...                                  # ✍️
    return float(sortino), float(calmar)

mine = [p.attempt(sortino_calmar, strats[s]) for s in strats]
mine = p.check("sortino and calmar", mine, [(p.tear_sheet(strats[s])["sortino"], p.tear_sheet(strats[s])["calmar"]) for s in strats])
pd.DataFrame({s: p.tear_sheet(strats[s]) for s in strats}).T.round(3)

In [ ]:
m = p.monthly_table(strats["S1"]) * 100
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(m.to_numpy(), cmap="RdBu", vmin=-6, vmax=6, aspect="auto")
ax.set_xticks(range(12), ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]); ax.set_yticks(range(len(m)), m.index)
ax.set_title("S1 monthly returns, % (true Sharpe 0.8)"); plt.colorbar(im); plt.show()
print(f"negative months: {(m < 0).to_numpy().sum()} of {m.notna().to_numpy().sum()}")

Compare each estimated Sharpe in the tear sheet with the true value it was generated with: ten years of data still misses by several tenths. Notebook 04 turns that into a standard error.

## Wrap-up

* Point-in-time universes, always; delisted names stay in the history.
* A tear sheet shows return, risk, drawdown depth and length, and tail shape, not just a Sharpe.
* Graded version: `labs/part08/week26_analysis` (plus trade statistics with MAE/MFE).